# 08c · 注意力机制与手写迷你 Transformer：化学反应产物预测

在 `08a` 和 `08b` 中，我们构建了仅含解码器（Decoder-only）的自回归语言模型（`SmilesRNN` / `SmilesLSTM`），从无条件分布 $P(x_1, \dots, x_T)$ 中采样生成新的有机光电分子。然而在合成化学与材料研发中，更常见的一类核心任务是**受条件约束的序列到序列转换（Sequence-to-Sequence, Seq2Seq）**：

- **正向反应产物预测（Forward Reaction Prediction）**：给定反应物（Reactants）与试剂（Reagents），预测主产物（Major Product）的分子结构；
- **逆合成路线规划（Single-Step Retrosynthesis）**：给定目标产物分子，反推可行的前驱体反应物组合。

如果沿用传统的 RNN 编码器-解码器（Encoder-Decoder）架构，编码器必须把长达数十个 token 的反应物 SMILES 压缩进**单个固定维度的隐状态向量** $\mathbf{h}_T$ 中，再由解码器从中还原出产物结构。对于有机化学反应而言，这种“单向量信息瓶颈”尤为致命：绝大多数反应只改变分子局部的 1~2 个化学键，而占分子主体 80% 以上的芳环骨架与取代基需要原封不动（或仅调整环闭合数字编号）地复制到产物中。

2017 年提出的 **Transformer** 架构彻底抛弃了循环递归，改用**注意力机制（Attention Mechanism）**让解码器在生成每一个产物 token 时，都能直接回顾并聚焦于反应物序列中的任意原子位置。本节我们将以有机光电材料合成中最经典的 **Suzuki-Miyaura 交叉偶联反应**为载体，对比基于专家规则的反应模板（SMARTS）与数据驱动的序列翻译范式，亲手实现缩放点积注意力、正弦位置编码与编码器-解码器 Transformer，并通过交叉注意力热力图（Cross-Attention Map）直观观察模型如何“定位”反应活性位点。

## 1. 化学反应预测的两种范式：反应模板 vs. 序列到序列翻译

在 `08a` 与 `08b` 探讨的噻吩类共轭聚合物及给体-受体（D-A）有机半导体合成中，**Suzuki-Miyaura 交叉偶联反应**是构筑芳基-芳基碳碳单键（$\text{Ar}_1\text{–Ar}_2$）最核心的工具：在钯（Pd）催化剂与碱的作用下，芳基卤代物（$\text{Ar}_1\text{–Br}$ 或 $\text{Ar}_1\text{–Cl}$）与芳基硼酸（$\text{Ar}_2\text{–B(OH)}_2$）发生脱卤-转金属-还原消除循环，偶联生成联芳基产物。

在计算机辅助合成设计（CASP）中，预测该反应产物主要有两种路径：

1. **基于专家规则的反应模板（Reaction SMARTS Templates）**：化学家把反应机理编码为子结构匹配与图编辑规则（例如 `[c:1][Br,Cl].[c:2]B(O)O>>[c:1]-[c:2]`），利用 RDKit 搜索反应物中的卤素与硼酸位点并拼接成键；
2. **无模板序列到序列翻译（Template-Free Seq2Seq Translation）**：把用点号 `.` 分隔的反应物 SMILES 视为“源语言句子”（如 `Brc1cc2ccccc2s1.CC(C)(C)c1csc(B(O)O)c1`），把产物 SMILES 视为“目标语言句子”（`CC(C)(C)c1csc(-c2cc3ccccc3s2)c1`），交由神经网络端到端学习。

我们先导入依赖库，并用 RDKit 演示反应模板法的工作方式与局限性。

In [ ]:
import os
import re
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, rdChemReactions

RDLogger.DisableLog("rdApp.*")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


In [ ]:
# 演示：用 RDKit Reaction SMARTS 模板执行 Suzuki 偶联预测
suzuki_smarts = "[c:1][Br,Cl].[c:2]B(O)O>>[c:1]-[c:2]"
rxn_rule = rdChemReactions.ReactionFromSmarts(suzuki_smarts)

sample_halide = "Brc1cc2ccccc2s1"            # 2-溴苯并[b]噻吩
sample_boronic = "CC(C)(C)c1csc(B(O)O)c1"    # 4-叔丁基噻吩-2-硼酸

halide_mol = Chem.MolFromSmiles(sample_halide)
boronic_mol = Chem.MolFromSmiles(sample_boronic)
products = rxn_rule.RunReactants((halide_mol, boronic_mol))

pred_smiles_set = set()
for prod_tuple in products:
    mol = prod_tuple[0]
    try:
        Chem.SanitizeMol(mol)
        pred_smiles_set.add(Chem.MolToSmiles(mol, canonical=True))
    except Exception:
        pass

print(f"Reactant 1 (Aryl Halide)  : {sample_halide}")
print(f"Reactant 2 (Boronic Acid) : {sample_boronic}")
print(f"SMARTS Predicted Product  : {list(pred_smiles_set)}")

对于上述单一活性位点的标准底物，SMARTS 模板给出了完全准确的产物。但当反应物中同时存在多个卤素（如既有 `-Br` 又有 `-Cl`，在 Suzuki 反应中 `-Br` 的氧化加成活性远高于 `-Cl`）、或者存在复杂杂环互变异构与立体电子效应时，死板的子结构模板会枚举出所有拓扑匹配的副产物，无法自主判断**区域选择性（Regioselectivity）**与**化学选择性（Chemoselectivity）**。

接下来我们加载本节使用的 `data/suzuki_toy.csv` 数据集，观察序列模型面临的数据格式与泛化挑战。

In [ ]:
DATA_DIR = "./data" if os.path.exists("./data/suzuki_toy.csv") else "course/08 - Sequence Models and Transformers/data"
suzuki_df = pd.read_csv(os.path.join(DATA_DIR, "suzuki_toy.csv"))

print(f"Total reactions: {len(suzuki_df):,}")
print("\nSplit counts:")
print(suzuki_df["split"].value_counts())

suzuki_df.head(5)

注意 `suzuki_df["split"]` 包含三个精心设计的子集，这与我们在 `02c` 中学习的**分子骨架划分（Scaffold Split）**一脉相承：

- **`train`（8,000 条）**：由常见芳基卤代物与芳基硼酸组合而成的训练集；
- **`val`（500 条，分布内新组合）**：卤代物与硼酸片段各自都在 `train` 中出现过，但它们的**配对组合**在训练集中从未见过；
- **`test_new_aryl`（500 条，分布外新芳环骨架）**：包含训练集中从未出现过的全新芳基卤代物骨架，专门检验模型是死记硬背了特定反应物词段，还是真正学会了“识别离去基团并拼接芳环”的反应规律。

仔细对比第一行数据的输入与输出：
- 输入 `reactants`：`Brc1cc2ccccc2s1.CC(C)(C)c1csc(B(O)O)c1`
- 输出 `product`：`CC(C)(C)c1csc(-c2cc3ccccc3s2)c1`

没有任何显式的原子映射编号或图连接矩阵，序列模型必须独立完成三件事：
1. 找到点号 `.` 两侧的离去基团 `Br` 与 `B(O)O` 并将其切除；
2. 在原来连接 `B(O)O` 的碳原子处插入芳香单键分支 `(-...)`；
3. 将第一个反应物原本的环闭合编号 `c1cc2ccccc2s1` 整体顺延重写为 `c2cc3ccccc3s2`，以避免与左侧噻吩环的 `1` 号环发生冲突！

In [ ]:
# 构建反应序列词表与张量编码
TOKEN_PATTERN = re.compile(
    r"(\[[^\]]+]|Br|Cl|Si|Se|Na|Li|Mg|Ca|Fe|Zn|Cu|Mn|@@|@|\(|\)|\.|=|#|-|\+|\\\\|\/|~|[0-9]|[A-Za-z])"
)

def tokenize_smiles(s):
    return TOKEN_PATTERN.findall(s)

all_reaction_texts = list(suzuki_df["reactants"]) + list(suzuki_df["product"])
raw_tokens = sorted({tok for s in all_reaction_texts for tok in tokenize_smiles(s)})
vocab = ["<pad>", "^", "$"] + raw_tokens
tok2idx = {t: i for i, t in enumerate(vocab)}
idx2tok = {i: t for i, t in enumerate(vocab)}

PAD_ID, BOS_ID, EOS_ID = tok2idx["<pad>"], tok2idx["^"], tok2idx["$"]
MAX_SRC_LEN = 64
MAX_TGT_LEN = 64

print(f"Reaction vocabulary size: {len(vocab)}")
print(f"Tokens: {vocab}")

def encode_seq(s, max_len=64):
    ids = [BOS_ID] + [tok2idx[t] for t in tokenize_smiles(s)][: max_len - 2] + [EOS_ID]
    return ids + [PAD_ID] * (max_len - len(ids))

def decode_ids(ids):
    out = []
    for idx in ids:
        idx = int(idx)
        if idx == EOS_ID:
            break
        if idx not in (PAD_ID, BOS_ID):
            out.append(idx2tok[idx])
    return "".join(out)

def build_split_tensors(sub_df):
    src = torch.tensor([encode_seq(s, MAX_SRC_LEN) for s in sub_df["reactants"]], dtype=torch.long)
    tgt = torch.tensor([encode_seq(s, MAX_TGT_LEN) for s in sub_df["product"]], dtype=torch.long)
    return src, tgt

train_df = suzuki_df[suzuki_df["split"] == "train"].reset_index(drop=True)
val_df = suzuki_df[suzuki_df["split"] == "val"].reset_index(drop=True)
test_aryl_df = suzuki_df[suzuki_df["split"] == "test_new_aryl"].reset_index(drop=True)

train_src, train_tgt = build_split_tensors(train_df)
val_src, val_tgt = build_split_tensors(val_df)
test_src, test_tgt = build_split_tensors(test_aryl_df)

print(f"train_src shape: {tuple(train_src.shape)}, train_tgt shape: {tuple(train_tgt.shape)}")

---
## 2. 拆解 Transformer 核心组件：缩放点积注意力与位置编码

### 2.1 缩放点积注意力（Scaled Dot-Product Attention）

为什么 Transformer 不需要像 LSTM 那样按时间步传递隐状态？答案在于它把序列交互抽象成了**可微分的字典检索**。对于序列中的每一个位置：
- **Query（查询向量 $\mathbf{q}_i \in \mathbb{R}^{d_k}$）**：代表当前位置正在寻找什么化学环境（例如解码器正在生成偶联单键 `-` 之后的原子，它发出查询：“反应物中哪个芳香碳原子刚脱去了 `Br`？”）；
- **Key（键向量 $\mathbf{k}_j \in \mathbb{R}^{d_k}$）**：代表被检索位置的特征标签；
- **Value（值向量 $\mathbf{v}_j \in \mathbb{R}^{d_v}$）**：代表一旦匹配成功，被检索位置向当前位置传递的上下文表示。

将所有位置的向量堆叠为矩阵 $\mathbf{Q} \in \mathbb{R}^{T_q \times d_k}$、$\mathbf{K} \in \mathbb{R}^{T_k \times d_k}$、$\mathbf{V} \in \mathbb{R}^{T_k \times d_v}$，**缩放点积注意力**定义为：

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}} + \mathbf{M}\right)\mathbf{V}$$

这里有两个关键细节：
1. **为什么要除以 $\sqrt{d_k}$？** 若 $\mathbf{q}$ 与 $\mathbf{k}$ 的各分量独立同分布且均值为 $0$、方差为 $1$，则它们的内积 $\mathbf{q}^\top \mathbf{k} = \sum_{m=1}^{d_k} q_m k_m$ 的均值为 $0$，方差会随维度线性增长至 $d_k$（标准差为 $\sqrt{d_k}$）。当 $d_k$ 较大时，未经缩放的内积绝对值很大，会把 Softmax 推入梯度接近 $0$ 的极度饱和区；除以 $\sqrt{d_k}$ 能将内积方差重新拉回 $1$。
2. **加性掩码矩阵 $\mathbf{M}$**：对于不允许关注的位置（例如 `<pad>` 填充符，或解码器自注意力中尚未生成的未来位置 $j > i$），令 $M_{ij} = -\infty$，经过 $\exp(-\infty) = 0$ 后，该位置的注意力权重严格归零。

下面我们用纯 PyTorch 编写 `scaled_dot_product_attention` 并验证其数值性质。

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    计算缩放点积注意力。
    Q: (..., T_q, d_k)
    K: (..., T_k, d_k)
    V: (..., T_k, d_v)
    mask: 布尔张量或加性掩码，形状可广播至 (..., T_q, T_k)。
          若为布尔张量，True 表示该位置被屏蔽（填入 -inf）。
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dtype == torch.bool:
            scores = scores.masked_fill(mask, float("-inf"))
        else:
            scores = scores + mask
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# 用 5 个化学 token 测试因果自注意力（Causal Self-Attention）
demo_tokens = ["^", "Brc1", "cc", "s1", "$"]
T_demo, d_k_demo = len(demo_tokens), 8
torch.manual_seed(7)
Q_demo = torch.randn(1, T_demo, d_k_demo)
K_demo = torch.randn(1, T_demo, d_k_demo)
V_demo = torch.randn(1, T_demo, d_k_demo)

# 上三角布尔掩码（diagonal=1）：位置 i 无法看到 j > i 的未来 token
causal_mask_demo = torch.triu(torch.ones(T_demo, T_demo, dtype=torch.bool), diagonal=1)
out_demo, weights_demo = scaled_dot_product_attention(Q_demo, K_demo, V_demo, mask=causal_mask_demo)

print("Causal Attention Weights (rows = query step t, cols = key step j <= t):")
print(np.round(weights_demo[0].numpy(), 3))

可以看到，注意力权重矩阵严格呈下三角分布：第 $0$ 行（`^`）只能关注自己（权重为 `1.0`），第 $2$ 行（`cc`）的权重分布在列 `0, 1, 2` 上且行和为 `1.0`，而所有上三角部分（$j > i$）严格为 `0.0`。

### 2.2 正弦位置编码（Sinusoidal Positional Encoding）

仔细审视上面的注意力公式可以发现：如果没有因果掩码，同时置换序列中任意两个 token 的位置，注意力输出也只是做完全相同的行置换——也就是说，**纯注意力机制本身对序列顺序是盲目的（置换等变，Permutation-Equivariant）**。但在 SMILES 中，`CC(=O)O`（乙酸）和 `O=C(C)O` 或原子顺序打乱后的字符串截然不同，环闭合数字与括号的相对位置更是决定了分子拓扑。

为此，我们在把词嵌入送入编码器与解码器之前，需要加上与位置索引 $pos \in \{0, 1, \dots, T-1\}$ 唯一对应的**正弦位置编码**：

$$PE_{(pos,\, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \qquad PE_{(pos,\, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

不同维度 $2i$ 对应从 $2\pi$ 到 $10000 \cdot 2\pi$ 呈几何级数递增的波长。由三角恒等式 $\sin(\alpha+\beta) = \sin\alpha\cos\beta + \cos\alpha\sin\beta$ 可知，位置 $pos + k$ 的编码向量可以表示为位置 $pos$ 编码向量的线性变换，这使模型能轻而易举地捕捉**相对位置间距** $k$。

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        return x + self.pe[:, : x.size(1)]

pos_enc = SinusoidalPositionalEncoding(d_model=64, max_len=64)
pe_matrix = pos_enc.pe[0].numpy()  # (64, 64)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
im0 = axes[0].imshow(pe_matrix, cmap="RdBu_r", aspect="auto")
axes[0].set_xlabel("Embedding Dimension (i)")
axes[0].set_ylabel("Token Position (pos)")
axes[0].set_title("Sinusoidal Positional Encoding Matrix")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# 计算位置 20 与所有其他位置的内积相似度
sim_to_20 = pe_matrix @ pe_matrix[20]
axes[1].plot(range(64), sim_to_20, color="#1f77b4", lw=2)
axes[1].axvline(20, color="#d62728", ls="--", lw=1.2, label="pos = 20")
axes[1].set_xlabel("Token Position (pos)")
axes[1].set_ylabel("Dot Product with PE[20]")
axes[1].set_title("Relative Distance Decay in Positional Encoding")
axes[1].legend()

plt.tight_layout()
plt.show()

### 练习 08c-1：验证解码器因果掩码防止未来信息泄露

在训练 Transformer 解码器时，最常见的新手 Bug 是忘记传入因果掩码（`causal_mask`）。如果没有因果掩码，当解码器在位置 $t$ 预测下一个目标 token $y_{t+1}$ 时，多头注意力可以直接在输入序列的位置 $t+1$ 看到标准答案，导致训练损失在几个迭代内迅速降到接近 $0$，但推理生成时却一塌糊涂。

请在下方实现函数 `check_causal_leakage(Q, K, V)`：
1. 分别计算**带因果掩码**与**不带掩码**时的缩放点积注意力输出 `out_masked` 和 `out_unmasked`；
2. 将输入序列最后一个位置的值向量 `V[:, -1, :]` 加上一个巨大的扰动（例如 `+ 100.0`），再次计算两者的输出；
3. 对比第 $0$ 个时间步（`t = 0`）的输出变化范数 $\|\Delta \text{out}_{t=0}\|_2$：验证带因果掩码时，未来位置 $T-1$ 的剧烈变化对 $t=0$ 的影响严格为 `0.0`，而不带掩码时则会发生严重的信息泄露。

In [ ]:
# 练习 08c-1：在此编写代码验证因果掩码对未来信息泄露的阻断作用
def check_causal_leakage(Q, K, V):
    # TODO:
    # 1. 构造上三角布尔掩码 causal_mask
    # 2. 克隆 V 并对最后一个时间步 V_perturbed[:, -1, :] += 100.0
    # 3. 比较带掩码与不带掩码在 t=0 处的输出差异范数
    pass


---
## 3. 构建与训练 ReactionTransformer：分布内组合 vs. 新芳环骨架外推

现在我们将编码器（Encoder）与解码器（Decoder）组装成完整的 `ReactionTransformer`。为了在下一节能够直接提取并可视化解码器对反应物的**交叉注意力（Cross-Attention）**权重，我们显式编写 `EncoderLayer` 和 `DecoderLayer`，并在 `DecoderLayer` 中将最新一次前向传播的交叉注意力图保存在 `self.last_cross_attn` 属性中（形状为 `(B, nhead, T_tgt, T_src)`）：

- **编码器层（`EncoderLayer`）**：双向自注意力（反应物中的每个原子可以同时看到左侧卤代物和右侧硼酸的所有原子） + 前馈网络（FFN）；
- **解码器层（`DecoderLayer`）**：
  1. **掩码自注意力（Masked Self-Attention）**：已生成的产物前缀彼此交互（受因果掩码约束）；
  2. **交叉注意力（Cross-Attention）**：以产物前缀作为 Query，去检索编码器输出的反应物记忆表示（Memory，作为 Key 和 Value）；
  3. **前馈网络（Position-wise FFN）**。

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None):
        attn_out, _ = self.self_attn(
            src, src, src, key_padding_mask=src_key_padding_mask, need_weights=False
        )
        src = self.norm1(src + self.drop(attn_out))
        ffn_out = self.ffn(src)
        src = self.norm2(src + self.drop(ffn_out))
        return src


class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.last_cross_attn = None

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        # 1. 因果掩码自注意力
        self_out, _ = self.self_attn(
            tgt, tgt, tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask,
            need_weights=False,
        )
        tgt = self.norm1(tgt + self.drop(self_out))

        # 2. 编码器-解码器交叉注意力（保留各注意力头的权重用于可视化）
        cross_out, cross_weights = self.cross_attn(
            tgt, memory, memory,
            key_padding_mask=memory_key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        self.last_cross_attn = cross_weights.detach()
        tgt = self.norm2(tgt + self.drop(cross_out))

        # 3. 逐位置前馈网络
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.drop(ffn_out))
        return tgt


class ReactionTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model, max_len=128)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, nhead, dim_ff, dropout) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, nhead, dim_ff, dropout) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def encode(self, src):
        src_pad_mask = (src == PAD_ID)
        memory = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        for layer in self.enc_layers:
            memory = layer(memory, src_key_padding_mask=src_pad_mask)
        return memory, src_pad_mask

    def decode(self, tgt_in, memory, src_pad_mask):
        T_tgt = tgt_in.size(1)
        tgt_mask = torch.triu(torch.ones(T_tgt, T_tgt, device=tgt_in.device, dtype=torch.bool), diagonal=1)
        tgt_pad_mask = (tgt_in == PAD_ID)
        h = self.pos_encoder(self.embedding(tgt_in) * math.sqrt(self.d_model))
        for layer in self.dec_layers:
            h = layer(
                h, memory,
                tgt_mask=tgt_mask,
                tgt_key_padding_mask=tgt_pad_mask,
                memory_key_padding_mask=src_pad_mask,
            )
        return self.fc_out(h)

    def forward(self, src, tgt_in):
        memory, src_pad_mask = self.encode(src)
        return self.decode(tgt_in, memory, src_pad_mask)

下面我们加载预训练好的 `suzuki_transformer.pt` 权重（如果本地尚未生成或你想亲手观察从头训练的过程，只需将 `TRAIN_FROM_SCRATCH = True`，在 CPU 上训练 15 个 Epoch 约需 40 秒），并实现批量贪心解码函数 `greedy_translate_batch` 来预测产物序列。

In [ ]:
model = ReactionTransformer(len(vocab), d_model=128, nhead=4, num_layers=2, dim_ff=256, dropout=0.1).to(device)
ckpt_path = os.path.join(DATA_DIR, "suzuki_transformer.pt")
TRAIN_FROM_SCRATCH = False

if os.path.exists(ckpt_path) and not TRAIN_FROM_SCRATCH:
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    history = ckpt.get("history", {})
    print(f"Loaded pretrained ReactionTransformer from {ckpt_path}")
else:
    print("Training ReactionTransformer from scratch...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    train_loader = DataLoader(TensorDataset(train_src, train_tgt), batch_size=128, shuffle=True)
    history = {"train_loss": [], "val_loss": []}
    for epoch in range(1, 16):
        model.train()
        total_loss = 0.0
        for b_src, b_tgt in train_loader:
            b_src, b_tgt = b_src.to(device), b_tgt.to(device)
            logits = model(b_src, b_tgt[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, len(vocab)), b_tgt[:, 1:].reshape(-1), ignore_index=PAD_ID)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * b_src.size(0)
        tr_loss = total_loss / len(train_src)
        model.eval()
        with torch.no_grad():
            v_logits = model(val_src.to(device), val_tgt[:, :-1].to(device))
            v_loss = F.cross_entropy(v_logits.reshape(-1, len(vocab)), val_tgt[:, 1:].to(device).reshape(-1), ignore_index=PAD_ID).item()
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(v_loss)
        if epoch == 1 or epoch % 3 == 0:
            print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={v_loss:.4f}")

In [ ]:
@torch.no_grad()
def greedy_translate_batch(model, src_tensor, max_len=MAX_TGT_LEN):
    """对一批反应物序列执行自回归贪心解码（T -> 0）。"""
    model.eval()
    src_tensor = src_tensor.to(device)
    B = src_tensor.size(0)
    memory, src_pad_mask = model.encode(src_tensor)

    ys = torch.full((B, 1), BOS_ID, dtype=torch.long, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for _ in range(max_len - 1):
        logits = model.decode(ys, memory, src_pad_mask)
        next_tok = logits[:, -1, :].argmax(dim=-1)
        next_tok = torch.where(finished, torch.tensor(PAD_ID, device=device), next_tok)
        ys = torch.cat([ys, next_tok.unsqueeze(1)], dim=1)
        finished = finished | (next_tok == EOS_ID)
        if finished.all():
            break

    return [decode_ids(row) for row in ys.cpu()]


def canon_smiles(s):
    if not s:
        return None
    mol = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None


def evaluate_reaction_predictions(model, src_tensor, gold_smiles_list, batch_size=128):
    preds = []
    for i in range(0, len(src_tensor), batch_size):
        preds.extend(greedy_translate_batch(model, src_tensor[i : i + batch_size]))

    n_valid = 0
    n_exact_str = 0
    n_exact_canon = 0
    for pred_s, gold_s in zip(preds, gold_smiles_list):
        if pred_s == gold_s:
            n_exact_str += 1
        c_pred = canon_smiles(pred_s)
        c_gold = canon_smiles(gold_s)
        if c_pred is not None:
            n_valid += 1
            if c_pred == c_gold:
                n_exact_canon += 1

    N = len(gold_smiles_list)
    return {
        "Validity (%)": 100.0 * n_valid / N,
        "Exact String Match (%)": 100.0 * n_exact_str / N,
        "Canonical Top-1 Acc (%)": 100.0 * n_exact_canon / N,
        "predictions": preds,
    }

val_res = evaluate_reaction_predictions(model, val_src, list(val_df["product"]))
test_res = evaluate_reaction_predictions(model, test_src, list(test_aryl_df["product"]))

summary_df = pd.DataFrame([
    {"Split": "val (In-Distribution New Combinations)", **{k: f"{v:.1f}" for k, v in val_res.items() if k != "predictions"}},
    {"Split": "test_new_aryl (OOD Unseen Aryl Halides)", **{k: f"{v:.1f}" for k, v in test_res.items() if k != "predictions"}},
])
summary_df

对比 `val`（分布内新组合）与 `test_new_aryl`（分布外全新芳基卤代物骨架）的评估结果，我们可以清楚地看到序列模型在化学反应预测中的核心特性：

1. **分布内组合泛化能力极强**：在 `val` 上，尽管具体的卤代物+硼酸配对从未在训练集中同时出现过，模型依然能以极高的语法有效率和 Top-1 准确率拼出正确的偶联产物并完成环编号重排；
2. **分布外新骨架外推退化（OOD Degradation）**：当面对 `test_new_aryl` 中训练阶段未见过的全新芳环骨架时，无模板序列模型在复制陌生骨架或重排多环闭合数字时更容易出错，导致精确匹配率明显下降。这与我们在 `02c` 中看到的属性预测模型在 Scaffold Split 下性能回落是同一本质：纯数据驱动的序列模型并不内置化学价态守恒定律，遇到未见过的子结构词段组合时可能产生局部拼写偏离。

### 练习 08c-2：区域选择性冲突与无模板模型的预测行为分析

当芳基卤代物上同时带有溴原子（`-Br`）和氯原子（`-Cl`）时（例如 `Brc1ccc(Cl)s1`，2-溴-5-氯噻吩），实验上钯催化的 Suzuki 偶联会优先在活性更高的 `C–Br` 键处发生反应，保留 `C–Cl` 键。

请在下方编写代码：
1. 将双卤代底物 `Brc1ccc(Cl)s1` 与硼酸 `CC(C)(C)c1csc(B(O)O)c1` 分别输入第 1 节的 **SMARTS 模板（`rxn_rule`）**与我们训练好的 **`ReactionTransformer`**；
2. 观察 SMARTS 模板给出了几个候选产物（为什么？），以及 `ReactionTransformer` 给出的 Top-1 预测产物保留了哪个卤素原子；
3. 从 `test_new_aryl` 中找出 3 个 `ReactionTransformer` 预测失败（`Canonical Top-1` 不匹配）的样本，打印其反应物、真实产物与模型预测产物，分析模型错在了哪个环节（是语法无效、卤素位点找错，还是陌生芳环骨架复制变形？）。

In [ ]:
# 练习 08c-2：在此对比双卤代底物的区域选择性预测，并诊断 test_new_aryl 上的失败样本
competing_rxn = "Brc1ccc(Cl)s1.CC(C)(C)c1csc(B(O)O)c1"

# TODO:
# 1. 用 rxn_rule.RunReactants 预测 competing_rxn 的所有匹配产物
# 2. 用 greedy_translate_batch 预测 competing_rxn 的 Top-1 产物
# 3. 检查 test_aryl_df 与 test_res["predictions"] 中不匹配的 3 个样本


---
## 4. 交叉注意力可视化与真实专利反应（USPTO）初探

### 4.1 交叉注意力热力图（Cross-Attention Map）：模型在看哪里？

在 `DecoderLayer` 的第 2 个子层中，解码器每生成一个产物 token（纵轴 Query），都会对编码器输出的全部反应物 token（横轴 Key）计算一组归一化注意力分布。我们取出最后一层解码器在预测一条验证集反应时的 `last_cross_attn` 张量，绘制其跨头平均注意力热力图。

In [ ]:
# 选取一条验证集反应，执行单步教师强制前向传播并提取最后一层 Cross-Attention
sample_idx = 0
src_str = val_df.loc[sample_idx, "reactants"]
tgt_str = val_df.loc[sample_idx, "product"]

src_tokens = ["^"] + tokenize_smiles(src_str) + ["$"]
tgt_tokens_in = ["^"] + tokenize_smiles(tgt_str)
tgt_tokens_out = tokenize_smiles(tgt_str) + ["$"]

src_ids = torch.tensor([[tok2idx[t] for t in src_tokens]], dtype=torch.long, device=device)
tgt_in_ids = torch.tensor([[tok2idx[t] for t in tgt_tokens_in]], dtype=torch.long, device=device)

model.eval()
with torch.no_grad():
    _ = model(src_ids, tgt_in_ids)
    # shape: (1, nhead, T_tgt, T_src)
    cross_attn = model.dec_layers[-1].last_cross_attn[0].cpu().numpy()

mean_cross_attn = cross_attn.mean(axis=0)  # (T_tgt, T_src)

fig, ax = plt.subplots(figsize=(11, 7.5))
im = ax.imshow(mean_cross_attn, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(src_tokens)))
ax.set_xticklabels(src_tokens, rotation=45, ha="right", fontsize=8.5)
ax.set_yticks(range(len(tgt_tokens_out)))
ax.set_yticklabels(tgt_tokens_out, fontsize=8.5)
ax.set_xlabel("Encoder Reactant Tokens (Key / Value)")
ax.set_ylabel("Decoder Predicted Product Tokens (Query)")
ax.set_title("Decoder Layer 2 Cross-Attention Map (Averaged Across Heads)")
plt.colorbar(im, ax=ax, fraction=0.035)
plt.tight_layout()
plt.show()

观察上方的交叉注意力热力图，你会发现两条清晰的**对角高亮轨迹**：
1. 在解码前半段（生成硼酸侧的芳环骨架 `CC(C)(C)c1csc...`），注意力沿着反应物右侧（`.` 之后的硼酸分子）呈单调对角线移动，逐个复制未参与反应的取代基与杂环原子；
2. 当解码器遇到偶联位点（原来连接 `B(O)O` 的位置）准备写入分支 `(-c2cc3ccccc3s2)` 时，注意力焦点突然从反应物右侧**跨越到反应物左侧的溴代苯并噻吩骨架**（紧邻 `Br` 的芳香碳原子），并沿着左侧芳环完成第二条对角线复制！

这直观印证了注意力机制如何克服 RNN 的单向量瓶颈：模型无需把整条反应物死记在单个向量里，而是把编码器输出当作可随时按位置索引的外部工作记忆，在复制骨架时顺着对应片段移动光标，在构建新化学键时跨分子跳转到活性位点。

### 4.2 从理想偶联到真实专利反应：探索 USPTO 数据集

在 `suzuki_toy.csv` 中，每条输入只包含两个干净的底物分子。但在真实的实验文献与美国专利局（USPTO）化学反应数据库中，反应记录往往混合了**底物（Reactants）**、**催化剂（Catalysts，如钯配合物）**、**无机碱（Bases，如碳酸钾、叔丁醇钠）**以及**有机溶剂（Solvents，如 DMF、THF、二氧六环）**。

我们加载 `data/uspto_test_sample.csv`（包含 300 条从 USPTO 真实专利中抽样的有机反应），观察真实世界反应序列的复杂性。

In [ ]:
uspto_df = pd.read_csv(os.path.join(DATA_DIR, "uspto_test_sample.csv"))
print(f"USPTO sample count: {len(uspto_df)}")
for i in range(3):
    print(f"\n--- Reaction #{i} ---")
    print(f"Precursors : {uspto_df.loc[i, 'precursors']}")
    print(f"Product    : {uspto_df.loc[i, 'product']}")

可以看到，在真实 USPTO 数据中：
1. `precursors` 往往由 3~6 个用 `.` 分隔的分子组成（例如第 0 条包含有机锡试剂 `CCCC[Sn](CCCC)(CCCC)c1cscn1`、溶剂 `CN(C)C=O` 即 DMF、底物二氯代杂环以及吗啉）；
2. 大量溶剂与催化剂的原子完全不会进入最终产物（即所谓“旁观物种” Spectator Species），但它们决定了反应条件与选择性；
3. 词表中还会出现 `[Sn]`、`[Pd]`、`[Cs+]`、`[K+]` 等金属与离子方括号 token。

工业界主流的 Molecular Transformer（Schwaller et al., *ACS Central Science*, 2019）正是基于我们上面编写的完全相同的编码器-解码器架构，只是将词表扩展到覆盖全部元素符号，并在百万级 USPTO 反应上训练了更深的层数（4~6 层，8 头注意力），从而无需显式区分反应物与试剂就能直接实现约 90% 的正向反应 Top-1 准确率。

### 练习 08c-3：解析 USPTO 真实反应的旁观试剂与原子守恒比例

在处理真实反应数据时，衡量一条反应记录中“有多少重原子真正进入了产物”的指标称为**重原子转化率（Heavy Atom Utilization / Conservation Ratio）**。

请在下方编写代码统计 `uspto_df` 的 300 条专利反应：
1. 对每条反应，按 `.` 拆分 `precursors`，统计每条反应包含的前驱体组分分子个数（Component Count）分布；
2. 利用 RDKit 计算每条反应中 `product` 的重原子总数（`mol.GetNumHeavyAtoms()`）除以 `precursors` 的重原子总数，绘制该比值的直方图；
3. 找出 `uspto_df` 中最常出现的 Top-10 前驱体分子（提示：先对每个由 `.` 分隔的分子做 Canonical SMILES 标准化，再统计出现频次），看看排在前列的分子在有机合成中分别扮演什么角色（溶剂、酸碱还是常见试剂？）。

In [ ]:
# 练习 08c-3：在此统计 USPTO 反应的组分个数、重原子转化率与高频试剂/溶剂
from collections import Counter

# TODO:
# 1. 统计每条 precursors 按 '.' 拆分的分子个数与重原子利用率 (Product Heavy Atoms / Precursors Heavy Atoms)
# 2. 统计所有拆分出的子分子（标准化后）的出现频次 Top-10


---
## 5. 本节小结与模块衔接

回顾整个第 08 讲（`08a` $\to$ `08b` $\to$ `08c`），我们完成了一条从无条件序列建模到有条件序列转换的完整脉络：

1. **序列表示与自回归生成（`08a`）**：通过化学正则分词将分子映射为离散 token 序列，对比了马尔可夫 Bigram 与 LSTM 语言模型，揭示了隐状态在跨越数十个字符追踪括号配对与环闭合编号时的核心作用，并掌握了采样温度 $T$ 对有效率与多样性的调控；
2. **数据增强、似然探针与迁移微调（`08b`）**：在真实异构数据库 `AqSolDB` 上引入 **SMILES 枚举数据增强**，使模型摆脱对单一标准遍历起点的依赖；利用每 token 平均负对数似然（`NLL per token`）度量分子在学习分布中的“自然度”，并通过两阶段**迁移学习**在仅百余条有机半导体样本上实现了定向化学空间聚焦；
3. **注意力机制与条件反应翻译（`08c`）**：从反应模板（SMARTS）的局限出发，亲手构建了缩放点积注意力、正弦位置编码与编码器-解码器 `ReactionTransformer`，对比了分布内新组合与分布外新骨架的泛化差异，并通过交叉注意力热力图验证了模型对反应位点的自动对齐能力。

在下一讲（**09 · 生成模型与材料逆向设计**）中，我们将跳出纯自回归或反应翻译的框架，进一步探索如何把离散的分子与晶体结构映射进**连续光滑的潜空间（Latent Space）**——借助变分自编码器（VAE）、生成对抗网络（GAN）与扩散模型（Diffusion Models），在潜空间中沿目标性质梯度直接搜索并解码出全新的候选材料。